# Aula 004 — Modelos Probabilísticos: Exercícios
## Classificador Naive Bayes e Rede Bayesiana — Conjunto Lentes de Contato

In [ ]:
import pandas as pd
import numpy as np
from fractions import Fraction

# ── Conjunto de Treino: Lentes de Contato ──
dados = pd.DataFrame({
    'Idade':         ['infantil','infantil','infantil','infantil',
                      'adolescente','adolescente','adolescente','adolescente','adolescente',
                      'adulto','adulto','adulto','adulto','adulto','adulto'],
    'Diagnostico':   ['miopia','miopia','hipermetropia','hipermetropia',
                      'miopia','miopia','miopia','hipermetropia','hipermetropia',
                      'miopia','miopia','miopia','hipermetropia','hipermetropia','hipermetropia'],
    'Astigmatismo':  ['não','sim','não','sim',
                      'não','sim','não','não','sim',
                      'não','sim','sim','não','sim','não'],
    'Taxa_lacrimal': ['reduzida','normal','normal','normal',
                      'reduzida','reduzida','normal','reduzida','normal',
                      'normal','normal','normal','reduzida','normal','normal'],
    'Tipo_lente':    ['nenhuma','gelatinosa','gelatinosa','dura',
                      'gelatinosa','nenhuma','dura','gelatinosa','dura',
                      'gelatinosa','dura','gelatinosa','nenhuma','gelatinosa','gelatinosa']
})

print(f"Total de amostras: {len(dados)}")
display(dados)

---
## Exercício 1 — Tabela de Contagens e Probabilidades Suavizadas (Naive Bayes, α=1)

**Suavização Laplaciana**: adiciona-se α=1 a cada contagem no numerador e α × (nº de categorias do atributo) ao denominador.

In [ ]:
alpha = 1
classes = dados['Tipo_lente'].unique()
atributos = ['Idade', 'Diagnostico', 'Astigmatismo', 'Taxa_lacrimal']
N = len(dados)
n_classes = len(classes)

# ── Probabilidades a priori P(classe) com suavização ──
print("="*60)
print("PROBABILIDADES A PRIORI P(classe) — com suavização")
print("="*60)
contagem_classes = dados['Tipo_lente'].value_counts()
for c in sorted(classes):
    count = contagem_classes[c]
    prob = (count + alpha) / (N + alpha * n_classes)
    print(f"  P({c:12s}) = ({count}+1) / ({N}+{n_classes}) = {count+alpha}/{N+n_classes} = {prob:.4f}")

# ── Tabelas de contagens e probabilidades condicionais por atributo ──
print("\n")
tabelas = {}  # armazena para uso posterior

for attr in atributos:
    valores = sorted(dados[attr].unique())
    n_vals = len(valores)
    print("="*60)
    print(f"Atributo: {attr}  (categorias: {valores})")
    print("="*60)

    # Contagens observadas
    header = f"{'Valor':<16s}" + "".join(f"{c:>12s}" for c in sorted(classes))
    print("\n  CONTAGENS OBSERVADAS")
    print(f"  {header}")
    print(f"  {'-'*len(header)}")
    for v in valores:
        row = f"  {v:<16s}"
        for c in sorted(classes):
            count = len(dados[(dados[attr] == v) & (dados['Tipo_lente'] == c)])
            row += f"{count:>12d}"
        print(row)

    # Probabilidades suavizadas
    print(f"\n  PROBABILIDADES SUAVIZADAS  P({attr}|classe) com α={alpha}")
    print(f"  {header}")
    print(f"  {'-'*len(header)}")
    for v in valores:
        row = f"  {v:<16s}"
        for c in sorted(classes):
            count = len(dados[(dados[attr] == v) & (dados['Tipo_lente'] == c)])
            n_c = contagem_classes[c]
            prob = (count + alpha) / (n_c + alpha * n_vals)
            tabelas[(attr, v, c)] = prob
            row += f"  {count+alpha}/{n_c+n_vals:<4d} ={prob:.4f}"
        print(row)
    print()

---
## Exercício 2 — Classificação do paciente com Naive Bayes

Paciente: **Idade=infantil, Diagnóstico=hipermetropia, Astigmatismo=não, Taxa lacrimal=reduzida**

Para cada classe $c$:  
$P(c|X) \propto P(c) \times \prod_i P(x_i|c)$

Depois normalizamos para que a soma dê 1.

In [ ]:
# Paciente a classificar
paciente = {
    'Idade': 'infantil',
    'Diagnostico': 'hipermetropia',
    'Astigmatismo': 'não',
    'Taxa_lacrimal': 'reduzida'
}

print("Paciente:", paciente)
print("\n" + "="*60)
print("CÁLCULO DE P(classe|paciente) — Naive Bayes")
print("="*60)

scores = {}
for c in sorted(classes):
    n_c = contagem_classes[c]
    p_c = (n_c + alpha) / (N + alpha * n_classes)
    produto = p_c
    print(f"\n  Classe: {c}")
    print(f"    P({c}) = {p_c:.4f}")
    for attr in atributos:
        v = paciente[attr]
        p = tabelas[(attr, v, c)]
        print(f"    P({attr}={v}|{c}) = {p:.4f}")
        produto *= p
    scores[c] = produto
    print(f"    => Score (não normalizado) = {produto:.6f}")

# Normalização
total = sum(scores.values())
print("\n" + "="*60)
print("PROBABILIDADES NORMALIZADAS")
print("="*60)
for c in sorted(classes):
    prob_norm = scores[c] / total
    print(f"  P({c:12s}|paciente) = {scores[c]:.6f} / {total:.6f} = {prob_norm:.4f}  ({prob_norm*100:.2f}%)")

melhor = max(scores, key=scores.get)
print(f"\n  >>> Classe prevista: {melhor} <<<")

---
## Exercício 3 — Rede Bayesiana

**Estrutura**: há dependência entre **Idade** e **Taxa Lacrimal** (Taxa Lacrimal depende de Idade).  
Todos os atributos dependem da classe (Tipo de lente).

### Estrutura da rede:
```
         Tipo_lente (classe)
        /     |          \
       v      v           v
    Idade  Diagnostico  Astigmatismo
       \                  
        v                 
     Taxa_lacrimal        
```

Ou seja:
- P(Idade | Tipo_lente)
- P(Diagnostico | Tipo_lente)
- P(Astigmatismo | Tipo_lente)
- **P(Taxa_lacrimal | Idade, Tipo_lente)** ← agora depende também de Idade!
- P(Tipo_lente)

In [ ]:
# ── Exercício 3: Rede Bayesiana ──
# As tabelas de P(Idade|classe), P(Diagnostico|classe) e P(Astigmatismo|classe)
# já foram calculadas no Exercício 1 e permanecem as mesmas.

# A diferença é que agora Taxa_lacrimal depende de Idade E da classe:
# P(Taxa_lacrimal | Idade, Tipo_lente)

print("="*70)
print("TABELA CONDICIONAL: P(Taxa_lacrimal | Idade, Tipo_lente) — com α=1")
print("="*70)

vals_taxa = sorted(dados['Taxa_lacrimal'].unique())
vals_idade = sorted(dados['Idade'].unique())
n_taxa = len(vals_taxa)  # 2 categorias: normal, reduzida

tabela_bn = {}  # (taxa, idade, classe) -> prob suavizada

for idade in vals_idade:
    print(f"\n  Idade = {idade}")
    header = f"  {'Taxa_lacrimal':<16s}" + "".join(f"{c:>14s}" for c in sorted(classes))
    print(header)
    print(f"  {'-'*len(header)}")
    for taxa in vals_taxa:
        row = f"  {taxa:<16s}"
        for c in sorted(classes):
            # Contagem: amostras com essa idade, taxa e classe
            count = len(dados[(dados['Idade'] == idade) &
                              (dados['Taxa_lacrimal'] == taxa) &
                              (dados['Tipo_lente'] == c)])
            # Denominador: amostras com essa idade e classe
            n_ic = len(dados[(dados['Idade'] == idade) &
                             (dados['Tipo_lente'] == c)])
            prob = (count + alpha) / (n_ic + alpha * n_taxa)
            tabela_bn[(taxa, idade, c)] = prob
            row += f"  {count+alpha}/{n_ic+n_taxa:<4d} ={prob:.4f}"
        print(row)

print("\n\nAs tabelas de P(Idade|classe), P(Diagnostico|classe) e P(Astigmatismo|classe)")
print("permanecem as mesmas do Exercício 1.")

### Resumo da estrutura da Rede Bayesiana

| Nó | Pais | Tabela |
|---|---|---|
| Tipo_lente | — | P(Tipo_lente) |
| Idade | Tipo_lente | P(Idade \| Tipo_lente) |
| Diagnóstico | Tipo_lente | P(Diagnóstico \| Tipo_lente) |
| Astigmatismo | Tipo_lente | P(Astigmatismo \| Tipo_lente) |
| Taxa_lacrimal | Idade, Tipo_lente | P(Taxa_lacrimal \| Idade, Tipo_lente) |

---
## Exercício 4 — Reclassificação usando a Rede Bayesiana

Mesmo paciente: **Idade=infantil, Diagnóstico=hipermetropia, Astigmatismo=não, Taxa lacrimal=reduzida**

Agora usamos:

$P(c|X) \propto P(c) \times P(\text{Idade}|c) \times P(\text{Diag}|c) \times P(\text{Astig}|c) \times P(\text{Taxa}|\text{Idade}, c)$

In [ ]:
print("Paciente:", paciente)
print("\n" + "="*60)
print("CÁLCULO COM REDE BAYESIANA")
print("="*60)

scores_bn = {}
for c in sorted(classes):
    n_c = contagem_classes[c]
    p_c = (n_c + alpha) / (N + alpha * n_classes)

    # Atributos independentes da classe (mesmos do Naive Bayes)
    p_idade = tabelas[('Idade', paciente['Idade'], c)]
    p_diag  = tabelas[('Diagnostico', paciente['Diagnostico'], c)]
    p_astig = tabelas[('Astigmatismo', paciente['Astigmatismo'], c)]

    # Taxa lacrimal agora depende de Idade E classe
    p_taxa = tabela_bn[(paciente['Taxa_lacrimal'], paciente['Idade'], c)]

    produto = p_c * p_idade * p_diag * p_astig * p_taxa
    scores_bn[c] = produto

    print(f"\n  Classe: {c}")
    print(f"    P({c}) = {p_c:.4f}")
    print(f"    P(Idade=infantil|{c}) = {p_idade:.4f}")
    print(f"    P(Diag=hipermetropia|{c}) = {p_diag:.4f}")
    print(f"    P(Astig=não|{c}) = {p_astig:.4f}")
    print(f"    P(Taxa=reduzida|Idade=infantil,{c}) = {p_taxa:.4f}  ← mudou!")
    print(f"    => Score = {produto:.6f}")

# Normalização
total_bn = sum(scores_bn.values())
print("\n" + "="*60)
print("PROBABILIDADES NORMALIZADAS (Rede Bayesiana)")
print("="*60)
for c in sorted(classes):
    prob_norm = scores_bn[c] / total_bn
    print(f"  P({c:12s}|paciente) = {prob_norm:.4f}  ({prob_norm*100:.2f}%)")

melhor_bn = max(scores_bn, key=scores_bn.get)
print(f"\n  >>> Classe prevista (Rede Bayesiana): {melhor_bn} <<<")

In [ ]:
# ── Comparação final ──
print("="*60)
print("COMPARAÇÃO: Naive Bayes vs Rede Bayesiana")
print("="*60)
print(f"  Naive Bayes:     {melhor}")
print(f"  Rede Bayesiana:  {melhor_bn}")
print("\nProbabilidades normalizadas:")
print(f"  {'Classe':<14s}  {'Naive Bayes':>12s}  {'Rede Bayes':>12s}")
for c in sorted(classes):
    nb = scores[c] / total
    bn = scores_bn[c] / total_bn
    print(f"  {c:<14s}  {nb:>11.4f}   {bn:>11.4f}")